In [ ]:
# -*- coding: utf-8 -*-

# ============================================================
# Two-source mixed validation - KMeans3 version
#
# Source A: original 113 samples
# Source B: second/environmental 36 samples
#
# Fixed split:
#   Training:   A 89 + B 24 = 113 samples
#   Validation: A 24 + B 12 = 36 samples
#
# Repeated validation:
#   Repeat the same split strategy 50 times.
#
# Quality grouping:
#   KMeans3 based on 7 measured quality indicators.
#
# Important:
#   In each split, quality scaler and KMeans3 are fitted only on training samples.
#   Validation labels are assigned by the training-fitted scaler and KMeans3 centers.
# ============================================================

import os
import re
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

KMEANS_N_CLUSTERS = 3

CLASS_VALUES = [0, 1, 2]
CLASS_NAME_MAP = {
    0: "Class I",
    1: "Class II",
    2: "Class III",
}
CLASS_NAMES = ["Class I", "Class II", "Class III"]


# ============================================================
# 0. Paths
# ============================================================

def find_project_root(start_path=None):
    """
    Locate the repository root by searching upward for the
    'Pea samples-new' input-data directory.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "Pea samples-new").is_dir():
            return candidate

    raise FileNotFoundError(
        "Cannot locate the project root. The repository must contain "
        "a folder named 'Pea samples-new', and Jupyter must be started "
        "from the repository or one of its subfolders."
    )


PROJECT_DIR = find_project_root()
DATA_BASE = PROJECT_DIR / "Pea samples-new"
MODEL_DIR = DATA_BASE / "Regression model"

# Existing raw/input files only
A_PATCH_CSV = DATA_BASE / "pea_patch_dataset.csv"
B_PATCH_CSV = MODEL_DIR / "flour_external_pea_patch_dataset.csv"
B_QUALITY_XLSX = MODEL_DIR / "pea quality-2025.xlsx"

required_input_files = [
    A_PATCH_CSV,
    B_PATCH_CSV,
    B_QUALITY_XLSX,
]

missing_input_files = [
    path for path in required_input_files if not path.is_file()
]

if missing_input_files:
    missing_text = "\n".join(str(path) for path in missing_input_files)
    raise FileNotFoundError(
        "The following required input files were not found:\n"
        f"{missing_text}"
    )

# Save formal analysis outputs under the repository root
OUT_DIR = PROJECT_DIR / "Pea_TwoSource_Mixed113_KMeans3_Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPEAT_OUT_DIR = PROJECT_DIR / "Pea_TwoSource_Mixed113_KMeans3_RepeatedValidation"
REPEAT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)
print("Source A patch CSV:", A_PATCH_CSV)
print("Source B patch CSV:", B_PATCH_CSV)
print("Source B quality Excel:", B_QUALITY_XLSX)
print("Fixed model output folder:", OUT_DIR)
print("Repeated validation output folder:", REPEAT_OUT_DIR)


# ============================================================
# 1. Quality indicators
# ============================================================

QUALITY_COLS = [
    "Protein content",
    "Moisture content",
    "Water uptaking capacity",
    "Water binding capacity",
    "Oil binding capacity",
    "Protein solubility",
    "Peak viscosity (RVA)",
]

print("\nQuality indicators used for KMeans3:")
for c in QUALITY_COLS:
    print(" -", c)


# ============================================================
# 2. Helper functions
# ============================================================

def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def detect_sample_id_column(df_input):
    candidates = [
        "sample_id", "Sample ID", "Sample_ID", "sample", "Sample",
        "ID", "id", "No.", "No"
    ]

    norm_map = {normalize_name(c): c for c in df_input.columns}

    for cand in candidates:
        key = normalize_name(cand)
        if key in norm_map:
            return norm_map[key]

    raise ValueError("Cannot detect sample ID column.")


def detect_wavelength_columns(df, expected_min=900, expected_max=2600):
    pairs = []

    for c in df.columns:
        nums = re.findall(r"\d+\.\d+|\d+", str(c))
        if nums:
            try:
                val = float(nums[-1])
                if expected_min <= val <= expected_max:
                    pairs.append((c, val))
            except Exception:
                pass

    pairs = sorted(pairs, key=lambda x: x[1])

    wl_cols = [x[0] for x in pairs]
    wavelengths = np.array([x[1] for x in pairs], dtype=float)

    return wl_cols, wavelengths


def align_external_to_training_wavelengths(ext_df, train_wl_cols, train_wavelengths):
    ext_wl_cols, ext_wavelengths = detect_wavelength_columns(ext_df)

    ext_map = {
        round(float(wl), 2): col
        for col, wl in zip(ext_wl_cols, ext_wavelengths)
    }

    aligned_ext_cols = []
    missing = []

    for train_col, train_wl in zip(train_wl_cols, train_wavelengths):
        key = round(float(train_wl), 2)

        if key in ext_map:
            aligned_ext_cols.append(ext_map[key])
        else:
            missing.append(train_wl)

    if len(missing) > 0:
        raise ValueError(
            f"Source B spectra are missing {len(missing)} Source A training wavelengths.\n"
            f"First missing wavelengths: {missing[:20]}\n"
            "You need wavelength alignment/interpolation before modeling."
        )

    return aligned_ext_cols


QUALITY_CANDIDATES = {
    "Protein content": [
        "Protein content", "protein content", "Protein", "protein"
    ],
    "Moisture content": [
        "Moisture content", "moisture content", "Moisture", "moisture"
    ],
    "Water uptaking capacity": [
        "Water uptaking capacity", "Water uptake capacity", "WUC",
        "water uptaking capacity", "water uptake capacity"
    ],
    "Water binding capacity": [
        "Water binding capacity", "WBC", "water binding capacity"
    ],
    "Oil binding capacity": [
        "Oil binding capacity", "OBC", "oil binding capacity"
    ],
    "Protein solubility": [
        "Protein solubility", "protein solubility", "Solubility", "solubility"
    ],
    "Peak viscosity (RVA)": [
        "Peak viscosity (RVA)", "Peak viscosity", "Peak viscosity (RVU)",
        "RVA", "peak viscosity", "peak_viscosity"
    ],
}


def detect_quality_columns(df, required_quality):
    norm_map = {normalize_name(c): c for c in df.columns}
    found = {}

    for q in required_quality:
        actual = None

        for cand in QUALITY_CANDIDATES[q]:
            key = normalize_name(cand)
            if key in norm_map:
                actual = norm_map[key]
                break

        if actual is not None:
            found[q] = actual

    return found


def load_quality_excel(xlsx_path, required_quality):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Quality Excel file not found: {xlsx_path}")

    xls = pd.ExcelFile(xlsx_path)

    for sheet in xls.sheet_names:
        temp = pd.read_excel(xlsx_path, sheet_name=sheet)
        qmap = detect_quality_columns(temp, required_quality)

        if all(q in qmap for q in required_quality):
            sid_col = detect_sample_id_column(temp)

            out = temp[[sid_col] + [qmap[q] for q in required_quality]].copy()
            out = out.rename(columns={sid_col: "sample_id"})

            rename_dict = {qmap[q]: q for q in required_quality}
            out = out.rename(columns=rename_dict)

            out["sample_id"] = pd.to_numeric(out["sample_id"], errors="coerce")
            out = out.dropna(subset=["sample_id"])
            out["sample_id"] = out["sample_id"].astype(int)

            out = out[["sample_id"] + required_quality]
            out = out.groupby("sample_id", as_index=False).median(numeric_only=True)

            return out

    raise ValueError(
        "Cannot find one sheet containing all required quality indicators:\n"
        + "\n".join(required_quality)
    )


def assign_b_treatment(sample_id):
    sid = int(sample_id)

    if 1 <= sid <= 12:
        return "Control"
    elif 13 <= sid <= 24:
        return "150% N"
    elif 25 <= sid <= 36:
        return "50% water"
    else:
        return "Unknown"


class SNVTransformer(BaseEstimator, TransformerMixin):
    """Standard Normal Variate correction applied row-wise."""
    def __init__(self, eps=1e-12):
        self.eps = eps

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, keepdims=True)
        std = np.where(std < self.eps, 1.0, std)
        return (X - mean) / std


def remap_kmeans_labels_by_protein(kmeans, quality_cols):
    """
    Ordered cluster labels:
    Class I   = lowest protein-center cluster
    Class II  = middle protein-center cluster
    Class III = highest protein-center cluster

    This only fixes arbitrary KMeans numbering.
    """
    centers = kmeans.cluster_centers_

    if "Protein content" in quality_cols:
        protein_idx = quality_cols.index("Protein content")
        order_score = centers[:, protein_idx]
    else:
        order_score = centers.mean(axis=1)

    raw_order = np.argsort(order_score)
    raw_to_ordered = {int(raw): int(new) for new, raw in enumerate(raw_order)}

    return raw_to_ordered


def aggregate_patch_probabilities_to_sample(patch_df, patch_proba, class_values):
    temp = pd.DataFrame({
        "global_sample_id": patch_df["global_sample_id"].values,
        "source": patch_df["source"].values,
        "sample_id": patch_df["sample_id"].values,
    })

    proba_cols = []

    for j, c in enumerate(class_values):
        col = f"proba_class_{c}"
        temp[col] = patch_proba[:, j]
        proba_cols.append(col)

    sample_pred = (
        temp.groupby(["global_sample_id", "source", "sample_id"])[proba_cols]
        .mean()
        .reset_index()
    )

    pred_idx = np.argmax(sample_pred[proba_cols].values, axis=1)
    sample_pred["y_pred"] = np.array(class_values)[pred_idx]
    sample_pred["prediction_confidence"] = sample_pred[proba_cols].max(axis=1)

    return sample_pred


def make_full_proba_matrix(model, X, class_values):
    """
    Convert predict_proba output into fixed class order.
    Useful when a model internally stores classes as a subset or different order.
    """
    proba_raw = model.predict_proba(X)
    model_classes = list(model.classes_)

    proba = np.zeros((proba_raw.shape[0], len(class_values)))

    for j, c in enumerate(model_classes):
        if int(c) in class_values:
            full_j = class_values.index(int(c))
            proba[:, full_j] = proba_raw[:, j]

    return proba


def evaluate_predictions(result_df):
    y_true = result_df["y_true"].values.astype(int)
    y_pred = result_df["y_pred"].values.astype(int)

    return {
        "n_samples": result_df["global_sample_id"].nunique(),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def create_two_source_split_for_repeat(repeat_id, random_state=42):
    """
    Create one mixed-source split.

    Training:
        A 89 + B 24

    Validation:
        A 24 + B 12

    B validation is balanced by treatment:
        4 Control + 4 150% N + 4 50% water
    """

    rng = np.random.default_rng(random_state + repeat_id)

    a_ids = np.array(sorted(quality_a["global_sample_id"].unique()))

    if len(a_ids) != 113:
        raise ValueError(f"Source A should contain 113 samples, but got {len(a_ids)}.")

    a_val_ids = rng.choice(a_ids, size=24, replace=False)
    a_train_ids = np.array([x for x in a_ids if x not in a_val_ids])

    b_val_ids = []
    b_train_ids = []

    for treatment in ["Control", "150% N", "50% water"]:
        ids_t = np.array(sorted(
            quality_b.loc[
                quality_b["treatment"] == treatment,
                "global_sample_id"
            ].unique()
        ))

        if len(ids_t) != 12:
            raise ValueError(
                f"Source B treatment {treatment} should contain 12 samples, "
                f"but got {len(ids_t)}."
            )

        val_t = rng.choice(ids_t, size=4, replace=False)
        train_t = np.array([x for x in ids_t if x not in val_t])

        b_val_ids.extend(val_t.tolist())
        b_train_ids.extend(train_t.tolist())

    train_ids_repeat = sorted(a_train_ids.tolist() + b_train_ids)
    val_ids_repeat = sorted(a_val_ids.tolist() + b_val_ids)

    if len(train_ids_repeat) != 113:
        raise ValueError(f"Training set should contain 113 samples, but got {len(train_ids_repeat)}.")

    if len(val_ids_repeat) != 36:
        raise ValueError(f"Validation set should contain 36 samples, but got {len(val_ids_repeat)}.")

    return train_ids_repeat, val_ids_repeat


def fit_kmeans3_and_assign_labels(train_ids_repeat, val_ids_repeat, repeat_id=0):
    """
    Fit scaler + KMeans3 on training quality data.
    Assign labels to both training and validation samples.
    """

    quality_train_repeat = quality_all[
        quality_all["global_sample_id"].isin(train_ids_repeat)
    ].copy()

    quality_val_repeat = quality_all[
        quality_all["global_sample_id"].isin(val_ids_repeat)
    ].copy()

    if quality_train_repeat["global_sample_id"].nunique() != 113:
        raise ValueError("Training quality table does not contain 113 samples.")

    if quality_val_repeat["global_sample_id"].nunique() != 36:
        raise ValueError("Validation quality table does not contain 36 samples.")

    scaler_repeat = StandardScaler()
    Q_train_scaled = scaler_repeat.fit_transform(
        quality_train_repeat[QUALITY_COLS].values
    )

    kmeans_repeat = KMeans(
        n_clusters=KMEANS_N_CLUSTERS,
        random_state=RANDOM_STATE + repeat_id,
        n_init=50
    )

    raw_train_labels = kmeans_repeat.fit_predict(Q_train_scaled)

    raw_to_ordered_repeat = remap_kmeans_labels_by_protein(
        kmeans=kmeans_repeat,
        quality_cols=QUALITY_COLS
    )

    ordered_train_labels = np.array([
        raw_to_ordered_repeat[int(x)]
        for x in raw_train_labels
    ])

    quality_train_repeat["cluster_raw"] = raw_train_labels
    quality_train_repeat["y_true"] = ordered_train_labels
    quality_train_repeat["class_name"] = quality_train_repeat["y_true"].map(CLASS_NAME_MAP)

    Q_val_scaled = scaler_repeat.transform(
        quality_val_repeat[QUALITY_COLS].values
    )

    raw_val_labels = kmeans_repeat.predict(Q_val_scaled)

    ordered_val_labels = np.array([
        raw_to_ordered_repeat[int(x)]
        for x in raw_val_labels
    ])

    quality_val_repeat["cluster_raw"] = raw_val_labels
    quality_val_repeat["y_true"] = ordered_val_labels
    quality_val_repeat["class_name"] = quality_val_repeat["y_true"].map(CLASS_NAME_MAP)

    return (
        quality_train_repeat,
        quality_val_repeat,
        scaler_repeat,
        kmeans_repeat,
        raw_to_ordered_repeat,
    )


# ============================================================
# 3. Load Source A spectra and quality data
# ============================================================

df_a_raw = pd.read_csv(A_PATCH_CSV)

if "sample_id" not in df_a_raw.columns:
    sid_col = detect_sample_id_column(df_a_raw)
    df_a_raw = df_a_raw.rename(columns={sid_col: "sample_id"})

df_a_raw["sample_id"] = pd.to_numeric(df_a_raw["sample_id"], errors="coerce")
df_a_raw = df_a_raw.dropna(subset=["sample_id"]).copy()
df_a_raw["sample_id"] = df_a_raw["sample_id"].astype(int)

wl_cols, wavelengths = detect_wavelength_columns(df_a_raw)

if len(wl_cols) == 0:
    raise ValueError("No wavelength columns detected in Source A patch CSV.")

qmap_a = detect_quality_columns(df_a_raw, QUALITY_COLS)

missing_a_q = [q for q in QUALITY_COLS if q not in qmap_a]
if len(missing_a_q) > 0:
    raise ValueError(
        "Source A patch CSV is missing quality columns:\n"
        + "\n".join(missing_a_q)
    )

quality_a = (
    df_a_raw
    .groupby("sample_id")[[qmap_a[q] for q in QUALITY_COLS]]
    .median()
    .reset_index()
)

quality_a = quality_a.rename(
    columns={qmap_a[q]: q for q in QUALITY_COLS}
)

quality_a = quality_a.dropna(subset=QUALITY_COLS).reset_index(drop=True)

quality_a["source"] = "A_original"
quality_a["global_sample_id"] = quality_a["sample_id"].apply(lambda x: f"A_{int(x)}")
quality_a["treatment"] = "A_original"

df_a_spectra = df_a_raw[["sample_id"] + wl_cols].copy()
df_a_spectra["source"] = "A_original"
df_a_spectra["global_sample_id"] = df_a_spectra["sample_id"].apply(lambda x: f"A_{int(x)}")
df_a_spectra["treatment"] = "A_original"

df_a_spectra = df_a_spectra[
    ["global_sample_id", "source", "sample_id", "treatment"] + wl_cols
].copy()

quality_a = quality_a[
    ["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS
].copy()

print("\nSource A:")
print("Patch rows:", df_a_spectra.shape[0])
print("Samples:", quality_a["global_sample_id"].nunique())
print("Wavelength columns:", len(wl_cols))
print("First wavelength:", wavelengths[0])
print("Last wavelength:", wavelengths[-1])
display(quality_a.head())


# ============================================================
# 4. Load Source B spectra and quality data
# ============================================================

if not B_PATCH_CSV.exists():
    raise FileNotFoundError(f"Source B patch CSV not found: {B_PATCH_CSV}")

df_b_raw = pd.read_csv(B_PATCH_CSV)

if "sample_id" not in df_b_raw.columns:
    sid_col = detect_sample_id_column(df_b_raw)
    df_b_raw = df_b_raw.rename(columns={sid_col: "sample_id"})

df_b_raw["sample_id"] = pd.to_numeric(df_b_raw["sample_id"], errors="coerce")
df_b_raw = df_b_raw.dropna(subset=["sample_id"]).copy()
df_b_raw["sample_id"] = df_b_raw["sample_id"].astype(int)

aligned_b_wl_cols = align_external_to_training_wavelengths(
    ext_df=df_b_raw,
    train_wl_cols=wl_cols,
    train_wavelengths=wavelengths,
)

df_b_spectra = df_b_raw[["sample_id"] + aligned_b_wl_cols].copy()

rename_b_wl = {
    b_col: a_col
    for b_col, a_col in zip(aligned_b_wl_cols, wl_cols)
}

df_b_spectra = df_b_spectra.rename(columns=rename_b_wl)

quality_b = load_quality_excel(
    xlsx_path=B_QUALITY_XLSX,
    required_quality=QUALITY_COLS,
)

quality_b = quality_b.dropna(subset=QUALITY_COLS).reset_index(drop=True)

quality_b["source"] = "B_second"
quality_b["global_sample_id"] = quality_b["sample_id"].apply(lambda x: f"B_{int(x)}")
quality_b["treatment"] = quality_b["sample_id"].apply(assign_b_treatment)

df_b_spectra["source"] = "B_second"
df_b_spectra["global_sample_id"] = df_b_spectra["sample_id"].apply(lambda x: f"B_{int(x)}")
df_b_spectra["treatment"] = df_b_spectra["sample_id"].apply(assign_b_treatment)

df_b_spectra = df_b_spectra[
    ["global_sample_id", "source", "sample_id", "treatment"] + wl_cols
].copy()

quality_b = quality_b[
    ["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS
].copy()

print("\nSource B:")
print("Patch rows:", df_b_spectra.shape[0])
print("Samples:", quality_b["global_sample_id"].nunique())
print("Treatment distribution:")
print(quality_b["treatment"].value_counts())
display(quality_b.head())


# ============================================================
# 5. Combine A + B
# ============================================================

df_all_spectra = pd.concat(
    [df_a_spectra, df_b_spectra],
    axis=0,
    ignore_index=True
)

quality_all = pd.concat(
    [quality_a, quality_b],
    axis=0,
    ignore_index=True
)

if quality_all["global_sample_id"].duplicated().any():
    dup = quality_all.loc[
        quality_all["global_sample_id"].duplicated(),
        "global_sample_id"
    ].tolist()
    raise ValueError(f"Duplicated global_sample_id found: {dup[:10]}")

print("\nCombined A + B:")
print("Total samples:", quality_all["global_sample_id"].nunique())
print("A samples:", quality_all.query("source == 'A_original'")["global_sample_id"].nunique())
print("B samples:", quality_all.query("source == 'B_second'")["global_sample_id"].nunique())
print("Total patch rows:", df_all_spectra.shape[0])


# ============================================================
# 6. Define classification models
# ============================================================

models = {
    "LogReg": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            multi_class="auto",
            random_state=RANDOM_STATE,
        )),
    ]),

    "SVM_RBF": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),

    "RandomForest": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),

    "ExtraTrees": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("clf", ExtraTreesClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
}

models["SoftVoting"] = VotingClassifier(
    estimators=[
        ("lr", clone(models["LogReg"])),
        ("svm", clone(models["SVM_RBF"])),
        ("et", clone(models["ExtraTrees"])),
    ],
    voting="soft",
    weights=[1, 1, 1],
    n_jobs=1,
)

print("\nModels:", list(models.keys()))


# ============================================================
# 7. Fixed split: build original mixed-source 113 model
# ============================================================

train_ids, val_ids = create_two_source_split_for_repeat(
    repeat_id=0,
    random_state=RANDOM_STATE
)

split_df = pd.DataFrame({
    "global_sample_id": train_ids + val_ids,
    "split": ["train"] * len(train_ids) + ["validation"] * len(val_ids),
})

split_df = split_df.merge(
    quality_all[["global_sample_id", "source", "sample_id", "treatment"]],
    on="global_sample_id",
    how="left"
)

print("\nFixed split summary:")
print(split_df.groupby(["split", "source"]).size())

print("\nFixed split B treatment distribution:")
print(split_df.query("source == 'B_second'").groupby(["split", "treatment"]).size())

split_df.to_csv(
    OUT_DIR / "fixed_split_train113_validation36_KMeans3.csv",
    index=False
)

(
    quality_train,
    quality_val,
    quality_scaler,
    kmeans3,
    raw_to_ordered,
) = fit_kmeans3_and_assign_labels(
    train_ids_repeat=train_ids,
    val_ids_repeat=val_ids,
    repeat_id=0,
)

print("\nTraining 113 KMeans3 class distribution:")
print(quality_train["class_name"].value_counts().sort_index())

print("\nTraining class distribution by source:")
print(quality_train.groupby(["source", "class_name"]).size())

print("\nValidation 36 reference class distribution:")
print(quality_val["class_name"].value_counts().sort_index())

print("\nValidation class distribution by source:")
print(quality_val.groupby(["source", "class_name"]).size())

cluster_profile = (
    quality_train
    .groupby("class_name")[QUALITY_COLS]
    .agg(["mean", "std", "count"])
)

print("\nTraining KMeans3 cluster profile:")
display(cluster_profile)

quality_train.to_csv(
    OUT_DIR / "training113_kmeans3_labels_7ind.csv",
    index=False
)

quality_val.to_csv(
    OUT_DIR / "validation36_reference_labels_from_training_kmeans3_7ind.csv",
    index=False
)

cluster_profile.to_csv(
    OUT_DIR / "training113_kmeans3_cluster_profile_7ind.csv"
)


# ============================================================
# 8. Internal GroupKFold on fixed training 113
# ============================================================

label_map_train = quality_train.set_index("global_sample_id")["y_true"].to_dict()

df_train_patches = df_all_spectra[
    df_all_spectra["global_sample_id"].isin(train_ids)
].copy()

df_train_patches["y"] = (
    df_train_patches["global_sample_id"]
    .map(label_map_train)
    .astype(int)
)

X_train_all = df_train_patches[wl_cols].values.astype(float)
y_train_all = df_train_patches["y"].values.astype(int)
groups_train = df_train_patches["global_sample_id"].values

print("\nFixed training patch data:")
print("Training samples:", df_train_patches["global_sample_id"].nunique())
print("Training patches:", df_train_patches.shape[0])
print("Training patch-level class distribution:")
print(pd.Series(y_train_all).value_counts().sort_index())

gkf = GroupKFold(n_splits=5)

internal_rows = []
oof_prediction_tables = {}

for model_name, model_template in models.items():
    print("\n" + "=" * 80)
    print("Internal GroupKFold model:", model_name)
    print("=" * 80)

    sample_pred_list = []

    for fold, (tr_idx, te_idx) in enumerate(
        gkf.split(X_train_all, y_train_all, groups=groups_train),
        start=1
    ):
        model = clone(model_template)

        X_tr, y_tr = X_train_all[tr_idx], y_train_all[tr_idx]
        X_te = X_train_all[te_idx]

        patch_test_df = df_train_patches.iloc[te_idx][
            ["global_sample_id", "source", "sample_id"]
        ].copy()

        model.fit(X_tr, y_tr)

        proba = make_full_proba_matrix(
            model=model,
            X=X_te,
            class_values=CLASS_VALUES,
        )

        fold_sample_pred = aggregate_patch_probabilities_to_sample(
            patch_df=patch_test_df,
            patch_proba=proba,
            class_values=CLASS_VALUES,
        )

        true_map_fold = (
            df_train_patches
            .iloc[te_idx]
            .groupby("global_sample_id")["y"]
            .first()
            .to_dict()
        )

        fold_sample_pred["y_true"] = (
            fold_sample_pred["global_sample_id"]
            .map(true_map_fold)
            .astype(int)
        )

        fold_sample_pred["fold"] = fold
        fold_sample_pred["model"] = model_name

        sample_pred_list.append(fold_sample_pred)

    oof_pred = pd.concat(sample_pred_list, axis=0, ignore_index=True)
    oof_prediction_tables[model_name] = oof_pred

    metrics = evaluate_predictions(oof_pred)
    metrics["model"] = model_name

    internal_rows.append(metrics)

    print(metrics)
    print(
        classification_report(
            oof_pred["y_true"],
            oof_pred["y_pred"],
            labels=CLASS_VALUES,
            target_names=CLASS_NAMES,
            digits=3,
            zero_division=0,
        )
    )

internal_summary = pd.DataFrame(internal_rows)
internal_summary = internal_summary.sort_values(
    ["macro_f1", "balanced_accuracy", "accuracy"],
    ascending=False
).reset_index(drop=True)

print("\nInternal GroupKFold summary for fixed mixed-source training 113 - KMeans3:")
display(internal_summary)

internal_summary.to_csv(
    OUT_DIR / "internal_GroupKFold_summary_training113_KMeans3_7ind.csv",
    index=False
)

for model_name, pred_table in oof_prediction_tables.items():
    pred_table.to_csv(
        OUT_DIR / f"internal_oof_predictions_{model_name}_training113_KMeans3_7ind.csv",
        index=False
    )

best_model_name = internal_summary.iloc[0]["model"]
print("\nBest internal model by macro-F1:", best_model_name)


# ============================================================
# 9. Train final fixed original model on mixed-source training 113
# ============================================================

final_model = clone(models[best_model_name])
final_model.fit(X_train_all, y_train_all)

print("\nFinal KMeans3 original model trained on mixed-source 113 samples.")
print("Best model:", best_model_name)
print("Training samples:", df_train_patches["global_sample_id"].nunique())
print("Training patches:", df_train_patches.shape[0])


# ============================================================
# 10. Fixed validation on reserved 36 samples
# ============================================================

df_val_patches = df_all_spectra[
    df_all_spectra["global_sample_id"].isin(val_ids)
].copy()

if df_val_patches["global_sample_id"].nunique() != 36:
    raise ValueError(
        f"Validation patch data should contain 36 samples, "
        f"but got {df_val_patches['global_sample_id'].nunique()}."
    )

X_val = df_val_patches[wl_cols].values.astype(float)

proba_val = make_full_proba_matrix(
    model=final_model,
    X=X_val,
    class_values=CLASS_VALUES,
)

validation_pred = aggregate_patch_probabilities_to_sample(
    patch_df=df_val_patches[["global_sample_id", "source", "sample_id"]].copy(),
    patch_proba=proba_val,
    class_values=CLASS_VALUES,
)

validation_pred["pred_class_name"] = validation_pred["y_pred"].map(CLASS_NAME_MAP)

validation_results = validation_pred.merge(
    quality_val[
        ["global_sample_id", "source", "sample_id", "treatment",
         "y_true", "class_name"] + QUALITY_COLS
    ],
    on=["global_sample_id", "source", "sample_id"],
    how="inner"
)

validation_results = validation_results.rename(columns={
    "class_name": "true_class_name"
})

validation_results["correct"] = (
    validation_results["y_pred"] == validation_results["y_true"]
)

print("\nFixed validation sample-level predictions:")
display(validation_results.sort_values(["source", "sample_id"]))

validation_results.to_csv(
    OUT_DIR / "validation36_predictions_fixed_KMeans3_7ind.csv",
    index=False
)

fixed_validation_metrics = evaluate_predictions(validation_results)

fixed_validation_metrics["model"] = best_model_name
fixed_validation_metrics["scenario"] = "fixed_mixed_source_train113_validation36_KMeans3_7ind"

fixed_validation_metrics_df = pd.DataFrame([fixed_validation_metrics])

print("\n" + "=" * 90)
print("Fixed validation performance: mixed-source training 113 -> reserved validation 36 - KMeans3")
print("=" * 90)
display(fixed_validation_metrics_df)

print("\nFixed validation classification report:")
print(
    classification_report(
        validation_results["y_true"],
        validation_results["y_pred"],
        labels=CLASS_VALUES,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

fixed_validation_metrics_df.to_csv(
    OUT_DIR / "validation36_metrics_fixed_KMeans3_7ind.csv",
    index=False
)

cm_fixed = confusion_matrix(
    validation_results["y_true"],
    validation_results["y_pred"],
    labels=CLASS_VALUES,
)

fig, ax = plt.subplots(figsize=(5.5, 4.8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_fixed,
    display_labels=CLASS_NAMES,
)

disp.plot(ax=ax, values_format="d", colorbar=False)

ax.set_title(
    "Fixed validation: KMeans3 mixed-source 113 -> 36\n"
    f"{best_model_name}: Acc={fixed_validation_metrics['accuracy']:.3f}, "
    f"Macro-F1={fixed_validation_metrics['macro_f1']:.3f}"
)

plt.tight_layout()

cm_fixed_path = OUT_DIR / "validation36_confusion_matrix_fixed_KMeans3_7ind.png"
plt.savefig(cm_fixed_path, dpi=300)
plt.show()

print("\nSaved fixed validation confusion matrix:")
print(cm_fixed_path)


# ============================================================
# 11. Save fixed KMeans3 model package
# ============================================================

model_package = {
    "analysis_type": "two-source mixed training model - KMeans3",
    "description": "Training set = 89 samples from Source A + 24 samples from Source B. KMeans3 was fitted only on training 113 samples using 7 quality indicators.",
    "random_state": RANDOM_STATE,
    "kmeans_n_clusters": KMEANS_N_CLUSTERS,
    "quality_cols": QUALITY_COLS,
    "wl_cols": wl_cols,
    "wavelengths": wavelengths,
    "quality_scaler": quality_scaler,
    "kmeans3": kmeans3,
    "raw_to_ordered": raw_to_ordered,
    "best_model_name": best_model_name,
    "final_model": final_model,
    "models_available": list(models.keys()),
    "train_ids": train_ids,
    "validation_ids_reserved": val_ids,
    "source_a_patch_csv": str(A_PATCH_CSV),
    "source_b_patch_csv": str(B_PATCH_CSV),
    "source_b_quality_xlsx": str(B_QUALITY_XLSX),
}

joblib.dump(
    model_package,
    OUT_DIR / "mixed_source_training113_KMeans3_model_package.joblib"
)

# The combined 149-sample quality table and sample index remain in memory.
# They are not written as separate intermediate input files.

print("\nSaved fixed KMeans3 model package:")
print(OUT_DIR / "mixed_source_training113_KMeans3_model_package.joblib")


# ============================================================
# 12. Repeated two-source holdout validation - KMeans3
# ============================================================

N_REPEATS = 50

MODEL_NAMES_TO_RUN = [
    "LogReg",
    "SVM_RBF",
    "RandomForest",
    "ExtraTrees",
    "SoftVoting",
]

print("\n" + "=" * 90)
print("Start repeated two-source holdout validation - KMeans3")
print("=" * 90)
print("Repeated validation output folder:", REPEAT_OUT_DIR)
print("Number of repeats:", N_REPEATS)
print("Models:", MODEL_NAMES_TO_RUN)


def run_one_repeat_one_model_kmeans3(repeat_id, model_name, model_template):
    train_ids_repeat, val_ids_repeat = create_two_source_split_for_repeat(
        repeat_id=repeat_id,
        random_state=RANDOM_STATE
    )

    (
        quality_train_repeat,
        quality_val_repeat,
        scaler_repeat,
        kmeans_repeat,
        raw_to_ordered_repeat,
    ) = fit_kmeans3_and_assign_labels(
        train_ids_repeat=train_ids_repeat,
        val_ids_repeat=val_ids_repeat,
        repeat_id=repeat_id,
    )

    label_map_train = quality_train_repeat.set_index("global_sample_id")["y_true"].to_dict()

    df_train_patches_repeat = df_all_spectra[
        df_all_spectra["global_sample_id"].isin(train_ids_repeat)
    ].copy()

    df_val_patches_repeat = df_all_spectra[
        df_all_spectra["global_sample_id"].isin(val_ids_repeat)
    ].copy()

    df_train_patches_repeat["y"] = (
        df_train_patches_repeat["global_sample_id"]
        .map(label_map_train)
        .astype(int)
    )

    X_train_repeat = df_train_patches_repeat[wl_cols].values.astype(float)
    y_train_repeat = df_train_patches_repeat["y"].values.astype(int)

    X_val_repeat = df_val_patches_repeat[wl_cols].values.astype(float)

    model = clone(model_template)
    model.fit(X_train_repeat, y_train_repeat)

    proba_val = make_full_proba_matrix(
        model=model,
        X=X_val_repeat,
        class_values=CLASS_VALUES,
    )

    validation_pred = aggregate_patch_probabilities_to_sample(
        patch_df=df_val_patches_repeat[
            ["global_sample_id", "source", "sample_id"]
        ].copy(),
        patch_proba=proba_val,
        class_values=CLASS_VALUES,
    )

    validation_pred["pred_class_name"] = validation_pred["y_pred"].map(CLASS_NAME_MAP)

    validation_results_repeat = validation_pred.merge(
        quality_val_repeat[
            ["global_sample_id", "source", "sample_id", "treatment",
             "y_true", "class_name"] + QUALITY_COLS
        ],
        on=["global_sample_id", "source", "sample_id"],
        how="inner"
    )

    validation_results_repeat = validation_results_repeat.rename(columns={
        "class_name": "true_class_name"
    })

    validation_results_repeat["correct"] = (
        validation_results_repeat["y_pred"] ==
        validation_results_repeat["y_true"]
    )

    validation_results_repeat["repeat"] = repeat_id
    validation_results_repeat["model"] = model_name

    metrics = evaluate_predictions(validation_results_repeat)

    metrics.update({
        "repeat": repeat_id,
        "model": model_name,
        "n_train_samples": len(train_ids_repeat),
        "n_validation_samples": len(val_ids_repeat),
        "train_A_n": sum([x.startswith("A_") for x in train_ids_repeat]),
        "train_B_n": sum([x.startswith("B_") for x in train_ids_repeat]),
        "validation_A_n": sum([x.startswith("A_") for x in val_ids_repeat]),
        "validation_B_n": sum([x.startswith("B_") for x in val_ids_repeat]),
        "train_Class_I_n": int((quality_train_repeat["y_true"] == 0).sum()),
        "train_Class_II_n": int((quality_train_repeat["y_true"] == 1).sum()),
        "train_Class_III_n": int((quality_train_repeat["y_true"] == 2).sum()),
        "validation_Class_I_n": int((quality_val_repeat["y_true"] == 0).sum()),
        "validation_Class_II_n": int((quality_val_repeat["y_true"] == 1).sum()),
        "validation_Class_III_n": int((quality_val_repeat["y_true"] == 2).sum()),
        "mean_prediction_confidence": validation_results_repeat["prediction_confidence"].mean(),
    })

    return metrics, validation_results_repeat, quality_train_repeat, quality_val_repeat


all_metrics = []
all_predictions = []
all_validation_labels = []

for model_name in MODEL_NAMES_TO_RUN:
    if model_name not in models:
        raise ValueError(
            f"{model_name} not found in models. Available models: {list(models.keys())}"
        )

    print("\n" + "=" * 90)
    print("Repeated validation model:", model_name)
    print("=" * 90)

    for repeat_id in range(N_REPEATS):
        metrics, validation_results_repeat, quality_train_repeat, quality_val_repeat = run_one_repeat_one_model_kmeans3(
            repeat_id=repeat_id,
            model_name=model_name,
            model_template=models[model_name],
        )

        all_metrics.append(metrics)
        all_predictions.append(validation_results_repeat)

        temp_val_labels = quality_val_repeat[
            ["global_sample_id", "source", "sample_id", "treatment",
             "y_true", "class_name"] + QUALITY_COLS
        ].copy()

        temp_val_labels["repeat"] = repeat_id
        temp_val_labels["model"] = model_name

        all_validation_labels.append(temp_val_labels)

        if (repeat_id + 1) % 10 == 0:
            print(f"Finished {repeat_id + 1}/{N_REPEATS} repeats for {model_name}")


metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)
validation_labels_df = pd.concat(all_validation_labels, axis=0, ignore_index=True)

metrics_path = REPEAT_OUT_DIR / "repeated_validation_metrics_each_repeat_KMeans3.csv"
predictions_path = REPEAT_OUT_DIR / "repeated_validation_sample_predictions_KMeans3.csv"
labels_path = REPEAT_OUT_DIR / "repeated_validation_reference_labels_KMeans3.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
validation_labels_df.to_csv(labels_path, index=False)

print("\nSaved raw repeated-validation outputs:")
print(metrics_path)
print(predictions_path)
print(labels_path)


# ============================================================
# 13. Repeated validation summary by model
# ============================================================

summary_by_model = (
    metrics_df
    .groupby("model")[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "mean_prediction_confidence",
            "validation_Class_I_n",
            "validation_Class_II_n",
            "validation_Class_III_n",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

summary_by_model.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary_by_model.columns
]

summary_by_model = summary_by_model.reset_index()

summary_by_model = summary_by_model.sort_values(
    ["macro_f1_mean", "balanced_accuracy_mean", "accuracy_mean"],
    ascending=False
).reset_index(drop=True)

summary_path = REPEAT_OUT_DIR / "repeated_validation_summary_by_model_KMeans3.csv"
summary_by_model.to_csv(summary_path, index=False)

print("\nRepeated validation summary by model - KMeans3:")
display(summary_by_model)

print("\nFormatted summary:")
for _, row in summary_by_model.iterrows():
    print(
        f"{row['model']}: "
        f"Accuracy={row['accuracy_mean']:.3f} ± {row['accuracy_std']:.3f}; "
        f"Balanced accuracy={row['balanced_accuracy_mean']:.3f} ± {row['balanced_accuracy_std']:.3f}; "
        f"Macro-F1={row['macro_f1_mean']:.3f} ± {row['macro_f1_std']:.3f}; "
        f"Weighted-F1={row['weighted_f1_mean']:.3f} ± {row['weighted_f1_std']:.3f}"
    )

print("\nSaved repeated validation model summary:")
print(summary_path)


# ============================================================
# 14. Best model aggregated confusion matrix
# ============================================================

best_repeat_model = summary_by_model.iloc[0]["model"]

print("\nBest repeated-validation model by mean macro-F1:", best_repeat_model)

best_predictions = predictions_df[
    predictions_df["model"] == best_repeat_model
].copy()

y_true_all = best_predictions["y_true"].values.astype(int)
y_pred_all = best_predictions["y_pred"].values.astype(int)

cm = confusion_matrix(
    y_true_all,
    y_pred_all,
    labels=CLASS_VALUES,
)

fig, ax = plt.subplots(figsize=(5.8, 5.0))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES,
)

disp.plot(ax=ax, values_format="d", colorbar=False)

ax.set_title(
    "Repeated two-source holdout validation - KMeans3\n"
    f"Best model: {best_repeat_model}"
)

plt.tight_layout()

cm_path = REPEAT_OUT_DIR / "best_model_aggregated_confusion_matrix_KMeans3.png"
plt.savefig(cm_path, dpi=300)
plt.show()

print("\nAggregated classification report for best KMeans3 model:")
print(
    classification_report(
        y_true_all,
        y_pred_all,
        labels=CLASS_VALUES,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

print("\nSaved confusion matrix:")
print(cm_path)


# ============================================================
# 15. Metrics by source for best model
# ============================================================

source_rows = []

for (repeat_id, source), sub in best_predictions.groupby(["repeat", "source"]):
    metrics_sub = evaluate_predictions(sub)

    row = {
        "repeat": repeat_id,
        "source": source,
        "n_samples": sub["global_sample_id"].nunique(),
        "accuracy": metrics_sub["accuracy"],
        "balanced_accuracy": metrics_sub["balanced_accuracy"],
        "macro_f1": metrics_sub["macro_f1"],
        "weighted_f1": metrics_sub["weighted_f1"],
        "mean_prediction_confidence": sub["prediction_confidence"].mean(),
    }

    source_rows.append(row)

source_metrics_df = pd.DataFrame(source_rows)

source_summary = (
    source_metrics_df
    .groupby("source")[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "mean_prediction_confidence",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

source_metrics_path = REPEAT_OUT_DIR / "best_model_metrics_by_source_each_repeat_KMeans3.csv"
source_summary_path = REPEAT_OUT_DIR / "best_model_metrics_by_source_summary_KMeans3.csv"

source_metrics_df.to_csv(source_metrics_path, index=False)
source_summary.to_csv(source_summary_path)

print("\nBest KMeans3 model metrics by source:")
display(source_summary)

print("\nSaved source-level metrics:")
print(source_metrics_path)
print(source_summary_path)


# ============================================================
# 16. Metrics by B treatment for best model
# ============================================================

treatment_rows = []

best_predictions_b = best_predictions[
    best_predictions["source"] == "B_second"
].copy()

for (repeat_id, treatment), sub in best_predictions_b.groupby(["repeat", "treatment"]):
    metrics_sub = evaluate_predictions(sub)

    row = {
        "repeat": repeat_id,
        "treatment": treatment,
        "n_samples": sub["global_sample_id"].nunique(),
        "accuracy": metrics_sub["accuracy"],
        "balanced_accuracy": metrics_sub["balanced_accuracy"],
        "macro_f1": metrics_sub["macro_f1"],
        "weighted_f1": metrics_sub["weighted_f1"],
        "mean_prediction_confidence": sub["prediction_confidence"].mean(),
    }

    treatment_rows.append(row)

treatment_metrics_df = pd.DataFrame(treatment_rows)

treatment_summary = (
    treatment_metrics_df
    .groupby("treatment")[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "mean_prediction_confidence",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

treatment_metrics_path = REPEAT_OUT_DIR / "best_model_B_treatment_metrics_each_repeat_KMeans3.csv"
treatment_summary_path = REPEAT_OUT_DIR / "best_model_B_treatment_metrics_summary_KMeans3.csv"

treatment_metrics_df.to_csv(treatment_metrics_path, index=False)
treatment_summary.to_csv(treatment_summary_path)

print("\nBest KMeans3 model metrics by B treatment:")
display(treatment_summary)

print("\nSaved B-treatment metrics:")
print(treatment_metrics_path)
print(treatment_summary_path)


# ============================================================
# 17. Plot metric distribution across repeats
# ============================================================

plot_model_df = metrics_df[
    metrics_df["model"] == best_repeat_model
].copy()

fig, ax = plt.subplots(figsize=(7, 5))

metric_names = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
data_to_plot = [plot_model_df[m].values for m in metric_names]

ax.boxplot(data_to_plot, labels=metric_names)

ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title(
    "Repeated two-source holdout validation - KMeans3\n"
    f"Best model: {best_repeat_model}, n={N_REPEATS}"
)

plt.xticks(rotation=20, ha="right")
plt.tight_layout()

boxplot_path = REPEAT_OUT_DIR / "best_model_metric_distribution_boxplot_KMeans3.png"
plt.savefig(boxplot_path, dpi=300)
plt.show()

print("\nSaved metric distribution plot:")
print(boxplot_path)


# ============================================================
# 18. Save text summary
# ============================================================

best_row = summary_by_model.iloc[0]

summary_txt = f"""
Repeated two-source holdout validation summary - KMeans3

Design:
- Total sample pool: Source A 113 + Source B 36 = 149 samples
- Each repeat training set: 89 Source A + 24 Source B = 113 samples
- Each repeat validation set: 24 Source A + 12 Source B = 36 samples
- Source B validation set was treatment-balanced:
  4 Control + 4 150% N + 4 50% water

Quality indicators:
{QUALITY_COLS}

KMeans:
- n_clusters = 3
- Quality scaler and KMeans3 were fitted only on the training 113 samples in each repeat.
- Validation samples were assigned reference labels using the training-fitted scaler and KMeans3 centers.
- Spectral classifiers were trained only on the training 113 samples.
- Validation performance was evaluated on the reserved 36 samples.

Class ordering:
- Class I, II, and III were ordered by the protein coordinate of the KMeans cluster centers.
- This ordering only removes arbitrary KMeans label numbering.

Number of repeats:
{N_REPEATS}

Best model:
{best_repeat_model}

Best model repeated-validation performance:
- Accuracy: {best_row['accuracy_mean']:.3f} ± {best_row['accuracy_std']:.3f}
- Balanced accuracy: {best_row['balanced_accuracy_mean']:.3f} ± {best_row['balanced_accuracy_std']:.3f}
- Macro-F1: {best_row['macro_f1_mean']:.3f} ± {best_row['macro_f1_std']:.3f}
- Weighted-F1: {best_row['weighted_f1_mean']:.3f} ± {best_row['weighted_f1_std']:.3f}
"""

summary_txt_path = REPEAT_OUT_DIR / "repeated_validation_summary_KMeans3.txt"

with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write(summary_txt)

print("\nSaved text summary:")
print(summary_txt_path)

print("\nKMeans3 analysis finished.")
print("Fixed model outputs saved to:")
print(OUT_DIR)
print("Repeated validation outputs saved to:")
print(REPEAT_OUT_DIR)